# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN['data']['payload']}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

In [5]:
%run w06_validation_audit.ipynb


Finding 1 (Finding #4 - "The Freshness Multiplier"):
365+ day content that was refreshed within the last 30 days shows a 3.2x health-score
boost (10.7 -> 34.5) and 57x more impressions (71 -> 4,039) than 365+ day content that
was not refreshed.

Methodology question:
Which pages get "refreshed" is an editorial decision, not a random assignment. Editors
plausibly pick older pages that already show some early recovery signal, keyword demand,
or business priority to refresh -- not a random sample of stale pages. So the label
"refreshed vs not" is entangled with whatever made a page worth refreshing in the first
place.

Validation question:
This is a before/after comparison on an editor-selected group, not a randomized
experiment, so it cannot rule out that the pages chosen for refresh were already on a
different trajectory before the refresh happened. The paper itself flags a related
version of this problem for the neighboring 361+ freshness bucket (283:1 ratio on only
1 declining page) 

## 1. Question

*The research question and the decision it supports.*

In [1]:
''' # Question

## Research question

Can historical search-performance and query signals be used to rank content pages that deserve human review for possible refresh or other content action?

## Decision

The work supports the decision of which content pages should receive attention first when content-review resources are limited.

## User of the output

The primary user is a content or SEO team member who reviews pages and decides whether further investigation or a refresh is appropriate.

## Action

The output is a ranked review queue containing a priority score, reason code, and suggested action.

## Cost of a wrong recommendation

A false positive can waste reviewer time on a page that does not need intervention. A false negative can cause a potentially declining page to be missed. Therefore, the model is used as decision support rather than automatic content management.'''

' # Question\n\n## Research question\n\nCan historical search-performance and query signals be used to rank content pages that deserve human review for possible refresh or other content action?\n\n## Decision\n\nThe work supports the decision of which content pages should receive attention first when content-review resources are limited.\n\n## User of the output\n\nThe primary user is a content or SEO team member who reviews pages and decides whether further investigation or a refresh is appropriate.\n\n## Action\n\nThe output is a ranked review queue containing a priority score, reason code, and suggested action.\n\n## Cost of a wrong recommendation\n\nA false positive can waste reviewer time on a page that does not need intervention. A false negative can cause a potentially declining page to be missed. Therefore, the model is used as decision support rather than automatic content management.'

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [1]:
''' ## 2. Data

This project uses the **FlyRank ML Internship warehouse release**. The main table used for the lane is `fact_search_daily_performance`, with supporting information from `dim_content` and `dim_client`. Query-level signals are taken from the 90-day query data used to construct the modeling features.

The analysis works with a **90-day historical window** to create content-level features. Daily search-performance records are aggregated to the content-item level using `content_hash_id`.

The model uses historical signals such as impressions, visible query count, rare-query share, anonymized-impression share, query concentration, and position volatility.

The final month of the warehouse release is treated as a sealed outcome period rather than a development period, because it represents the natural future window for outcome-based analysis. Development and feature decisions are therefore based on an earlier period.

Label-derived fields such as `trend_direction`, `trend_pct`, and `is_declining_label` are excluded from the model features because they contain or directly derive information about the outcome and could cause leakage.

Client names, URLs, private search queries, and other identifying information are also excluded from the public-facing analysis. The published work uses the pseudonymized identifiers supplied by the dataset.

The purpose of this data is to support a public-safe, directional analysis of content-review prioritization rather than to expose individual client or search data.'''

' ## 2. Data\n\nThis project uses the **FlyRank ML Internship warehouse release**. The main table used for the lane is `fact_search_daily_performance`, with supporting information from `dim_content` and `dim_client`. Query-level signals are taken from the 90-day query data used to construct the modeling features.\n\nThe analysis works with a **90-day historical window** to create content-level features. Daily search-performance records are aggregated to the content-item level using `content_hash_id`.\n\nThe model uses historical signals such as impressions, visible query count, rare-query share, anonymized-impression share, query concentration, and position volatility.\n\nThe final month of the warehouse release is treated as a sealed outcome period rather than a development period, because it represents the natural future window for outcome-based analysis. Development and feature decisions are therefore based on an earlier period.\n\nLabel-derived fields such as `trend_direction`, `tr

In [6]:
print("Rows:", len(data_90d))
print("Features:", feature_cols_90d)
print("Start/end information is based on the warehouse window used to construct the 90-day features.")

Rows: 95895
Features: ['imp_mid30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share', 'pos_volatility_last30']
Start/end information is based on the warehouse window used to construct the 90-day features.


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [9]:
'''# Methodology

## Task type

This is a scoring and ranking problem with a binary declining-performance proxy used for model evaluation.

## Label / proxy

The analysis defines a declining-performance outcome using historical impression behavior:

`is_declining = 1` when impressions in the most recent 30-day period are less than 80% of impressions in the preceding 30-day period.

This is a constructed proxy for prioritization and should not be interpreted as a universal definition of content decline.

## Features

The model uses:

- `imp_mid30`
- `visible_queries`
- `rare_share`
- `anon_share`
- `top_query_share`
- `pos_volatility_last30`

These features describe historical search visibility, query coverage, query concentration, and ranking behavior.

## Baseline

The model is compared with the Week-4 baseline on the same evaluation task and metric.

The baseline is intentionally simple and transparent so that the model must demonstrate useful improvement rather than complexity for its own sake.

## Model

A Random Forest classifier was selected because it can capture nonlinear relationships between the available search signals and the constructed declining-performance outcome while remaining practical to interpret using feature importance and error analysis.

## Validation

The final audit uses a client-grouped split. Pages from the same client are kept entirely within either the training or testing partition.

This provides a more conservative estimate of generalization to unseen clients.

## Leakage checks

Label-derived fields such as `is_declining`, `trend_direction`, and `trend_pct` were excluded from the model features.

The feature timing was also reviewed to ensure that the selected signals represent information available before the content-review decision.'''

'# Methodology\n\n## Task type\n\nThis is a scoring and ranking problem with a binary declining-performance proxy used for model evaluation.\n\n## Label / proxy\n\nThe analysis defines a declining-performance outcome using historical impression behavior:\n\n`is_declining = 1` when impressions in the most recent 30-day period are less than 80% of impressions in the preceding 30-day period.\n\nThis is a constructed proxy for prioritization and should not be interpreted as a universal definition of content decline.\n\n## Features\n\nThe model uses:\n\n- `imp_mid30`\n- `visible_queries`\n- `rare_share`\n- `anon_share`\n- `top_query_share`\n- `pos_volatility_last30`\n\nThese features describe historical search visibility, query coverage, query concentration, and ranking behavior.\n\n## Baseline\n\nThe model is compared with the Week-4 baseline on the same evaluation task and metric.\n\nThe baseline is intentionally simple and transparent so that the model must demonstrate useful improvement

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [10]:
'''## 4. Results (vs baseline)

The model was evaluated against the Week-4 baseline using the same task and evaluation metric.

The baseline achieved a **Precision@50 of 0.240**, meaning that 24% of the top 50 pages selected by the baseline matched the defined declining-performance outcome.

The Random Forest achieved a **Precision@50 of 0.740**, meaning that 74% of the top 50 pages selected by the model matched the defined declining-performance outcome.

This is a substantial improvement over the baseline. The model therefore provides a more useful ranking of pages for the defined content-review task than the simple baseline on this evaluation.

However, Precision@50 should not be interpreted as 74% prediction accuracy for all pages. It only measures how many of the model's **top 50 ranked pages** matched the defined outcome.

The improvement is approximately **3.1 times the baseline precision** (0.740 / 0.240). This indicates that the model's ranking signals are useful for prioritizing a smaller set of pages for human review.

The Week-6 client-grouped validation provides a more conservative test because pages from the same client are kept out of both training and testing simultaneously. Therefore, the grouped result is the more appropriate result to use when discussing generalization to unseen clients.

Overall, the results provide evidence that the selected historical search and query signals can support content-review prioritization. They do not prove that these signals cause content decline, and they do not predict Google's ranking algorithm.'''

"## 4. Results (vs baseline)\n\nThe model was evaluated against the Week-4 baseline using the same task and evaluation metric.\n\nThe baseline achieved a **Precision@50 of 0.240**, meaning that 24% of the top 50 pages selected by the baseline matched the defined declining-performance outcome.\n\nThe Random Forest achieved a **Precision@50 of 0.740**, meaning that 74% of the top 50 pages selected by the model matched the defined declining-performance outcome.\n\nThis is a substantial improvement over the baseline. The model therefore provides a more useful ranking of pages for the defined content-review task than the simple baseline on this evaluation.\n\nHowever, Precision@50 should not be interpreted as 74% prediction accuracy for all pages. It only measures how many of the model's **top 50 ranked pages** matched the defined outcome.\n\nThe improvement is approximately **3.1 times the baseline precision** (0.740 / 0.240). This indicates that the model's ranking signals are useful for 

## 5. Limitations

*What this work cannot claim.*

In [11]:
'''# Limitations and Honest Framing

This analysis has several limitations.

First, the declining-performance label is a constructed proxy based on historical impression changes. It is not a ground-truth measure of whether content truly needs a refresh.

Second, the analysis is observational. The model identifies patterns associated with the defined outcome but does not establish causal relationships.

Third, the model does not predict Google's ranking algorithm or future search behavior.

Fourth, performance may vary across clients, topics, content types, and future time periods. The client-grouped validation reduces one source of optimistic evaluation but does not eliminate all forms of distribution shift.

Fifth, a high model score does not mean that a page should automatically be changed. A human reviewer must inspect the page and its context before taking action.

Therefore, the results should be described as observed, measured, directional, and useful for decision support rather than causal proof or guaranteed prediction.'''

"# Limitations and Honest Framing\n\nThis analysis has several limitations.\n\nFirst, the declining-performance label is a constructed proxy based on historical impression changes. It is not a ground-truth measure of whether content truly needs a refresh.\n\nSecond, the analysis is observational. The model identifies patterns associated with the defined outcome but does not establish causal relationships.\n\nThird, the model does not predict Google's ranking algorithm or future search behavior.\n\nFourth, performance may vary across clients, topics, content types, and future time periods. The client-grouped validation reduces one source of optimistic evaluation but does not eliminate all forms of distribution shift.\n\nFifth, a high model score does not mean that a page should automatically be changed. A human reviewer must inspect the page and its context before taking action.\n\nTherefore, the results should be described as observed, measured, directional, and useful for decision sup

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [12]:
'''# Ranked Recommendations

The model output should be used to create a prioritized content-review queue.

## Priority 1 — Review declining impressions

Reason code: `DECLINING_IMPRESSIONS`

Action: Review whether the content remains accurate, useful, and aligned with current search intent. Consider a refresh only when human review supports it.

## Priority 2 — Review ranking volatility

Reason code: `POSITION_VOLATILITY`

Action: Investigate changes in search-result positioning and whether the page still matches the relevant search intent.

## Priority 3 — Review query concentration

Reason code: `QUERY_CONCENTRATION`

Action: Review whether the page depends too heavily on a small number of queries and whether relevant query coverage can be improved.

## Priority 4 — Review low query visibility

Reason code: `LOW_QUERY_VISIBILITY`

Action: Review targeting, discoverability, and alignment with the intended search topic.

## Priority 5 — Manual review

Reason code: `MODEL_PRIORITY`

Action: Inspect the page manually before recommending any change.

These recommendations are prioritization rules, not automatic content changes.'''

'# Ranked Recommendations\n\nThe model output should be used to create a prioritized content-review queue.\n\n## Priority 1 — Review declining impressions\n\nReason code: `DECLINING_IMPRESSIONS`\n\nAction: Review whether the content remains accurate, useful, and aligned with current search intent. Consider a refresh only when human review supports it.\n\n## Priority 2 — Review ranking volatility\n\nReason code: `POSITION_VOLATILITY`\n\nAction: Investigate changes in search-result positioning and whether the page still matches the relevant search intent.\n\n## Priority 3 — Review query concentration\n\nReason code: `QUERY_CONCENTRATION`\n\nAction: Review whether the page depends too heavily on a small number of queries and whether relevant query coverage can be improved.\n\n## Priority 4 — Review low query visibility\n\nReason code: `LOW_QUERY_VISIBILITY`\n\nAction: Review targeting, discoverability, and alignment with the intended search topic.\n\n## Priority 5 — Manual review\n\nReaso

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [14]:
# Artifacts

'''The research artifact is supported by the following reproducible outputs:

- Week-1 research framing notebook
- Week-2 ML task framing notebook
- Week-3 data contract notebook
- Week-4 baseline notebook
- Week-5 model notebook
- Week-6 validation audit notebook
- Week-7 action playbook notebook
- Ranked action queue
- Model comparison results
- Review-priority figure

All source notebooks and code are available in the project repository.'''

'The research artifact is supported by the following reproducible outputs:\n\n- Week-1 research framing notebook\n- Week-2 ML task framing notebook\n- Week-3 data contract notebook\n- Week-4 baseline notebook\n- Week-5 model notebook\n- Week-6 validation audit notebook\n- Week-7 action playbook notebook\n- Ranked action queue\n- Model comparison results\n- Review-priority figure\n\nAll source notebooks and code are available in the project repository.'

In [15]:
'''# Reproducibility

The complete analysis is available in the project repository.

The notebooks document the progression from research question and data contract through baseline modeling, validation, and the final action playbook.

The main workflow is:

W01 → Research question
W02 → ML task framing
W03 → Data contract and leakage check
W04 → Baseline
W05 → Model
W06 → Validation audit
W07 → Action playbook
W08 → Research paper'''

'# Reproducibility\n\nThe complete analysis is available in the project repository.\n\nThe notebooks document the progression from research question and data contract through baseline modeling, validation, and the final action playbook.\n\nThe main workflow is:\n\nW01 → Research question  \nW02 → ML task framing  \nW03 → Data contract and leakage check  \nW04 → Baseline  \nW05 → Model  \nW06 → Validation audit  \nW07 → Action playbook  \nW08 → Research paper'

In [16]:
# Acknowledgments & Data Credit

'''Built on the FlyRank ML Internship dataset.

Data source: [FlyRank](https://flyrank.ai)

I thank the FlyRank ML Internship team for providing the dataset, learning framework, and project structure used for this work.'''

'Built on the FlyRank ML Internship dataset.\n\nData source: [FlyRank](https://flyrank.ai)\n\nI thank the FlyRank ML Internship team for providing the dataset, learning framework, and project structure used for this work.'

ABSTRACT

In [17]:
'''# Abstract

This project investigates whether historical search-performance and query signals can help prioritize content pages for human review and possible refresh. Using the FlyRank ML Internship dataset, I constructed historical features describing impressions, query visibility, query concentration, and ranking behavior, then compared a Random Forest model with a transparent baseline. The model was evaluated using client-grouped validation to reduce optimistic estimates caused by having pages from the same client in both training and testing data. The results show that these signals can provide useful directional decision support for prioritizing content review, although performance varies under stricter validation and the constructed declining label is only a proxy. The final output is a ranked review queue with reason codes and suggested actions, intended to support human content decisions rather than automate publishing or claim causal effects.'''

'# Abstract\n\nThis project investigates whether historical search-performance and query signals can help prioritize content pages for human review and possible refresh. Using the FlyRank ML Internship dataset, I constructed historical features describing impressions, query visibility, query concentration, and ranking behavior, then compared a Random Forest model with a transparent baseline. The model was evaluated using client-grouped validation to reduce optimistic estimates caused by having pages from the same client in both training and testing data. The results show that these signals can provide useful directional decision support for prioritizing content review, although performance varies under stricter validation and the constructed declining label is only a proxy. The final output is a ranked review queue with reason codes and suggested actions, intended to support human content decisions rather than automate publishing or claim causal effects.'

PROBLEM STATEMENT

In [18]:
'''# Introduction / Problem Statement

Content teams often have more pages to review than they can investigate manually. A useful system should therefore help identify which pages deserve attention first.

The problem addressed in this project is whether historical search-performance signals can be used to prioritize pages for content review.

The goal is not to automatically decide which pages should be changed. Instead, the goal is to produce a ranked queue that helps a human reviewer allocate limited time.

The research question is:

**Can historical search-performance and query signals provide useful decision support for prioritizing content pages for review?**'''

'# Introduction / Problem Statement\n\nContent teams often have more pages to review than they can investigate manually. A useful system should therefore help identify which pages deserve attention first.\n\nThe problem addressed in this project is whether historical search-performance signals can be used to prioritize pages for content review.\n\nThe goal is not to automatically decide which pages should be changed. Instead, the goal is to produce a ranked queue that helps a human reviewer allocate limited time.\n\nThe research question is:\n\n**Can historical search-performance and query signals provide useful decision support for prioritizing content pages for review?**'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.